# Create McKnight Awards (McKnight Foundation, research grants)

**Research-only filter.** Most McKnight Foundation grantmaking is not research
(Arts & Culture, Midwest Climate & Energy, Vibrant & Equitable Communities,
Minnesota Initiative Foundations, Director-Advised; searchable at
mcknight.org/grants/search-our-grants as org-level grants with no investigators).
This notebook ingests only McKnight's research grant program: the
**Collaborative Crop Research Program (CCRP)**, now "Global Collaboration for
Resilient Food Systems", which funds agroecology research by universities and
research organisations in Africa and the Andes. Every CCRP grant has a public
page at https://www.ccrp.org/grants/<slug>/ (McKnight-run WordPress site, all
grant pages listed in its Yoast sitemap) with title, lead organisation, partner
organisations, community of practice, countries, duration (MM/YYYY-MM/YYYY),
overview, grant aims and outputs/outcomes.

**414 CCRP grants, start years 2001-2025** (99.8% titled, 99% dated, 99.5% with lead organisation) (local run 2026-09-30).

**Not included: McKnight Endowment Fund for Neuroscience (MEFN).** MEFN's
Scholar / Technology / Memory & Cognitive Disorders / Neurobiology of Brain
Disorders awardees (1977-2026) are listed on
mcknight.org/programs/the-mcknight-endowment-fund-for-neuroscience/*/awardees/,
but MEFN is a separate OpenAlex funder (**F4320306144**, McKnight Endowment Fund
for Neuroscience). In the mcknight.org grants database McKnight Foundation
itself shows a single $38M grant to MEFN, which re-grants it. Those awards
belong under F4320306144 and are left for a separate ingest (flagged to the
batch coordinator), not blanket-assigned to McKnight Foundation.

**No PIs, amounts or grant numbers:** ccrp.org publishes none. The lead
organisation is kept as `lead_investigator.affiliation.name` with NULL person
names (HHMI/Helmsley/Mott precedent); amount/currency are NULL (Step 6.7 waived).
Untitled grant pages (published with an empty title) get a composed
`display_name` "McKnight CCRP grant: <lead org> (<start year>)"
(`title_is_composed = 'true'`).

**`funder_award_id` (section 2.1.1):** citing works write McKnight's internal
CCRP grant numbers ("15-473", "CCRP 16-098"; ~105 `crossref_work.grants` shells),
but no public McKnight/CCRP page prints them, so they cannot be mapped. The
award id is `CCRP-<WordPress post id>` of the grant page (stable, unique, in the
page markup as `postid-N`). Our awards therefore will not collapse onto those
shells.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320308324`
- display_name: McKnight Foundation
- ror_id / doi: from `openalex.funders.funders` (ror 003ghvj67, doi 10.13039/100005270)

**Prerequisites:** run `scripts/local/mcknight_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/mcknight/mcknight_projects.parquet` (section 1.4 shrink guard).

**Priority:** `497` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mcknight_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/mcknight/mcknight_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_award_ids FROM openalex.awards.mcknight_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.mcknight_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.mcknight_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320308324 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320308324;

## Step 2: Create McKnight Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mcknight_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320308324  -- McKnight Foundation
),

g AS (
    SELECT
        *,
        -- ccrp.org names no PI: keep the lead organisation as the award's
        -- affiliation anchor with NULL person names (HHMI/Helmsley/Mott precedent).
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
        CASE WHEN NULLIF(TRIM(lead_organization), '') IS NOT NULL THEN named_struct(
            'given_name', CAST(NULL AS STRING),
            'family_name', CAST(NULL AS STRING),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(lead_organization), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead
    FROM openalex.awards.mcknight_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.display_title), '') as display_name,
    COALESCE(NULLIF(TRIM(g.overview), ''), NULLIF(TRIM(g.grant_aims), '')) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,      -- ccrp.org publishes no per-grant amounts
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    'Collaborative Crop Research Program' as funder_scheme,
    'mcknight_ccrp_grants' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    g.lead as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (section 2.1.1): every funder_award_id is CCRP-<wordpress post id>.
SELECT assert_true(
    COUNT_IF(NOT funder_award_id RLIKE '^CCRP-[0-9]+$') = 0
    AND COUNT(*) = COUNT(DISTINCT id),
    'mcknight_awards: unexpected funder_award_id shape or duplicate ids'
) AS shape_ok
FROM openalex.awards.mcknight_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mcknight_ccrp_grants' AND priority = 497;

-- Insert into openalex_awards_raw with priority.
-- Priority 497: direct-from-funder ingest of McKnight's CCRP grant pages.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    497 as priority
FROM openalex.awards.mcknight_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_mcknight_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.mcknight_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, doi
FROM openalex.awards.mcknight_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, COUNT(description) as has_description
FROM openalex.awards.mcknight_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, COUNT(description) as has_description
FROM openalex.awards.mcknight_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check (no PIs published): lead organisations.
SELECT lead_investigator.affiliation.name AS lead_org, COUNT(*) AS n
FROM openalex.awards.mcknight_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.mcknight_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(doi) as has_doi,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.mcknight_awards;

In [ ]:
%sql

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mcknight_ccrp_grants'
GROUP BY provenance, priority;